# 🤖 DOM ML Prediction - Model Training

**Train 4 deep learning models for order book prediction:**
- LSTM (baseline)
- Bi-LSTM (primary candidate)
- GRU (speed-optimized)
- Conv1D (pattern recognition)

**Target:** Predict price direction (UP/DOWN/NEUTRAL) for BTCUSD and XAUUSD

---

### 📋 Instructions
1. Enable GPU: Runtime → Change runtime type → T4 GPU
2. Run cells in order
3. Download trained models at the end

## 1. Setup & Dependencies

In [ ]:
# Install dependencies
!pip install -q tensorflow pandas numpy scikit-learn python-binance ta

import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded!")

## 2. Configuration

In [ ]:
# Symbol Configuration
SYMBOL = 'BTCUSD'  # @param ['BTCUSD', 'XAUUSD']

SYMBOL_CONFIGS = {
    'BTCUSD': {
        'binance_symbol': 'BTCUSDT',
        'sequence_length': 100,
        'prediction_horizon': 10,  # seconds
        'batch_size': 256,
        'epochs': 100,
        'learning_rate': 0.001,
    },
    'XAUUSD': {
        'binance_symbol': 'PAXGUSDT',
        'sequence_length': 150,
        'prediction_horizon': 10,
        'batch_size': 128,
        'epochs': 120,
        'learning_rate': 0.0005,
    }
}

config = SYMBOL_CONFIGS[SYMBOL]
print(f"Training config for {SYMBOL}:")
print(config)

## 3. Fetch Historical Data from Binance

In [ ]:
from binance.client import Client
import time

# Initialize Binance client (no API key needed for public data)
client = Client()

def fetch_klines(symbol, interval='1m', days=30):
    """Fetch historical kline/candlestick data from Binance"""
    print(f"Fetching {days} days of {symbol} data...")
    
    start_time = datetime.now() - timedelta(days=days)
    start_str = start_time.strftime('%d %b %Y')
    
    klines = client.get_historical_klines(
        symbol=symbol,
        interval=interval,
        start_str=start_str
    )
    
    # Convert to DataFrame
    df = pd.DataFrame(klines, columns=[
        'open_time', 'open', 'high', 'low', 'close', 'volume',
        'close_time', 'quote_volume', 'trades', 'taker_buy_base',
        'taker_buy_quote', 'ignore'
    ])
    
    # Convert types
    df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
    for col in ['open', 'high', 'low', 'close', 'volume', 'quote_volume']:
        df[col] = df[col].astype(float)
    df['trades'] = df['trades'].astype(int)
    
    print(f"Fetched {len(df)} candles from {df['open_time'].iloc[0]} to {df['open_time'].iloc[-1]}")
    return df

# Fetch data
df_raw = fetch_klines(config['binance_symbol'], interval='1m', days=60)
df_raw.tail()

## 4. Feature Engineering

In [ ]:
def engineer_features(df):
    """Create ML features from OHLCV data"""
    df = df.copy()
    
    # Price features
    df['returns_1m'] = df['close'].pct_change() * 10000  # bps
    df['returns_5m'] = df['close'].pct_change(5) * 10000
    df['returns_15m'] = df['close'].pct_change(15) * 10000
    
    df['high_low_range'] = (df['high'] - df['low']) / df['close'] * 10000
    df['close_position'] = (df['close'] - df['low']) / (df['high'] - df['low'] + 1e-8)
    
    # Momentum
    df['momentum_10'] = df['close'] - df['close'].shift(10)
    df['momentum_20'] = df['close'] - df['close'].shift(20)
    
    # Volatility
    df['volatility_10m'] = df['returns_1m'].rolling(10).std()
    df['volatility_30m'] = df['returns_1m'].rolling(30).std()
    
    # Volume features
    df['volume_ma_10'] = df['volume'].rolling(10).mean()
    df['volume_ratio'] = df['volume'] / (df['volume_ma_10'] + 1e-8)
    df['volume_change'] = df['volume'].pct_change()
    
    # Buy/Sell pressure proxy (using taker data if available)
    if 'taker_buy_base' in df.columns:
        df['taker_buy_base'] = df['taker_buy_base'].astype(float)
        df['buy_ratio'] = df['taker_buy_base'] / (df['volume'] + 1e-8)
        df['buy_sell_imbalance'] = (df['buy_ratio'] - 0.5) * 200  # -100 to +100
    else:
        df['buy_sell_imbalance'] = 0
    
    # Moving averages
    df['ma_10'] = df['close'].rolling(10).mean()
    df['ma_30'] = df['close'].rolling(30).mean()
    df['ma_distance'] = (df['close'] - df['ma_10']) / df['close'] * 10000
    df['ma_cross'] = (df['ma_10'] - df['ma_30']) / df['close'] * 10000
    
    # RSI
    delta = df['close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    rs = gain / (loss + 1e-8)
    df['rsi'] = 100 - (100 / (1 + rs))
    
    # Time features
    df['hour'] = df['open_time'].dt.hour
    df['day_of_week'] = df['open_time'].dt.dayofweek
    
    # Target: Next period direction
    future_returns = df['close'].shift(-config['prediction_horizon']) / df['close'] - 1
    future_returns_bps = future_returns * 10000
    
    # Classify: DOWN (-1), NEUTRAL (0), UP (+1)
    threshold = 5  # 5 bps threshold
    df['target'] = 1  # Default neutral
    df.loc[future_returns_bps > threshold, 'target'] = 2  # UP
    df.loc[future_returns_bps < -threshold, 'target'] = 0  # DOWN
    
    # Drop NaN
    df = df.dropna()
    
    return df

# Engineer features
df = engineer_features(df_raw)
print(f"Dataset size: {len(df)} samples")
print(f"\nTarget distribution:")
print(df['target'].value_counts(normalize=True))

In [ ]:
# Define feature columns
FEATURE_COLS = [
    'returns_1m', 'returns_5m', 'returns_15m',
    'high_low_range', 'close_position',
    'momentum_10', 'momentum_20',
    'volatility_10m', 'volatility_30m',
    'volume_ratio', 'volume_change',
    'buy_sell_imbalance',
    'ma_distance', 'ma_cross',
    'rsi', 'hour', 'day_of_week'
]

print(f"Using {len(FEATURE_COLS)} features: {FEATURE_COLS}")

## 5. Prepare Training Data

In [ ]:
from sklearn.preprocessing import StandardScaler

def create_sequences(data, target, seq_length):
    """Create sequences for LSTM/GRU training"""
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i + seq_length])
        y.append(target[i + seq_length])
    return np.array(X), np.array(y)

# Extract features and target
features = df[FEATURE_COLS].values
target = df['target'].values

# Normalize features
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Create sequences
seq_length = config['sequence_length']
X, y = create_sequences(features_scaled, target, seq_length)

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")

# Time-series split (70% train, 15% val, 15% test)
n = len(X)
train_end = int(n * 0.7)
val_end = int(n * 0.85)

X_train, y_train = X[:train_end], y[:train_end]
X_val, y_val = X[train_end:val_end], y[train_end:val_end]
X_test, y_test = X[val_end:], y[val_end:]

print(f"\nTrain: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

## 6. Define Model Architectures

In [ ]:
def create_lstm_model(seq_length, n_features, n_classes=3):
    """Baseline LSTM"""
    inputs = layers.Input(shape=(seq_length, n_features))
    x = layers.LSTM(64, return_sequences=True)(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(32)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(16, activation='relu')(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    model = Model(inputs, outputs, name='lstm')
    return model

def create_bilstm_model(seq_length, n_features, n_classes=3):
    """Bidirectional LSTM"""
    inputs = layers.Input(shape=(seq_length, n_features))
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.Bidirectional(layers.LSTM(32))(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dense(16, activation='relu')(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    model = Model(inputs, outputs, name='bilstm')
    return model

def create_gru_model(seq_length, n_features, n_classes=3):
    """GRU (faster)"""
    inputs = layers.Input(shape=(seq_length, n_features))
    x = layers.GRU(64, return_sequences=True)(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.GRU(32)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(16, activation='relu')(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    model = Model(inputs, outputs, name='gru')
    return model

def create_conv1d_model(seq_length, n_features, n_classes=3):
    """Conv1D for pattern recognition"""
    inputs = layers.Input(shape=(seq_length, n_features))
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(64, 3, padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)
    model = Model(inputs, outputs, name='conv1d')
    return model

# Test models
n_features = X_train.shape[2]
for create_fn in [create_lstm_model, create_bilstm_model, create_gru_model, create_conv1d_model]:
    model = create_fn(seq_length, n_features)
    print(f"{model.name}: {model.count_params():,} parameters")

## 7. Train All Models

In [ ]:
def train_model(create_fn, name, X_train, y_train, X_val, y_val):
    """Train a single model"""
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")
    
    model = create_fn(seq_length, n_features)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['learning_rate']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=config['epochs'],
        batch_size=config['batch_size'],
        callbacks=callbacks,
        verbose=1
    )
    
    return model, history

# Train all models
models = {}
histories = {}

model_creators = [
    (create_lstm_model, 'lstm'),
    (create_bilstm_model, 'bilstm'),
    (create_gru_model, 'gru'),
    (create_conv1d_model, 'conv1d'),
]

for create_fn, name in model_creators:
    model, history = train_model(create_fn, name, X_train, y_train, X_val, y_val)
    models[name] = model
    histories[name] = history.history

print("\n✅ All models trained!")

## 8. Evaluate Models

In [ ]:
# Evaluate all models on test set
results = {}

print("\n📊 Model Evaluation Results:")
print("="*60)

for name, model in models.items():
    loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
    
    # Get predictions
    y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
    
    # Calculate F1
    from sklearn.metrics import f1_score
    f1 = f1_score(y_test, y_pred, average='weighted')
    
    results[name] = {
        'accuracy': accuracy,
        'f1_score': f1,
        'loss': loss
    }
    
    print(f"{name:10} | Accuracy: {accuracy:.4f} | F1: {f1:.4f}")

# Find best model
best_model = max(results, key=lambda k: results[k]['f1_score'])
print(f"\n🏆 Best Model: {best_model} (F1: {results[best_model]['f1_score']:.4f})")

In [ ]:
# Detailed report for best model
best = models[best_model]
y_pred = best.predict(X_test, verbose=0).argmax(axis=1)

print(f"\n📋 Classification Report for {best_model}:")
print(classification_report(y_test, y_pred, target_names=['DOWN', 'NEUTRAL', 'UP']))

# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['DOWN', 'NEUTRAL', 'UP'],
            yticklabels=['DOWN', 'NEUTRAL', 'UP'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix - {best_model}')
plt.show()

## 9. Learning Curves

In [ ]:
# Plot learning curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (name, history) in enumerate(histories.items()):
    ax = axes[idx // 2, idx % 2]
    ax.plot(history['accuracy'], label='Train')
    ax.plot(history['val_accuracy'], label='Validation')
    ax.set_title(f'{name} - Accuracy')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 10. Save Models

In [ ]:
import json
import pickle

# Create output directory
!mkdir -p models

# Save each model
for name, model in models.items():
    model.save(f'models/{name}_{SYMBOL}.keras')
    print(f"Saved: models/{name}_{SYMBOL}.keras")

# Save scaler
with open(f'models/scaler_{SYMBOL}.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print(f"Saved: models/scaler_{SYMBOL}.pkl")

# Save config
save_config = {
    'symbol': SYMBOL,
    'feature_cols': FEATURE_COLS,
    'sequence_length': seq_length,
    'results': results,
    'best_model': best_model
}
with open(f'models/config_{SYMBOL}.json', 'w') as f:
    json.dump(save_config, f, indent=2)
print(f"Saved: models/config_{SYMBOL}.json")

print("\n✅ All models saved!")

In [ ]:
# Download models (creates a zip)
!zip -r models_{SYMBOL}.zip models/

from google.colab import files
files.download(f'models_{SYMBOL}.zip')

print(f"\n📥 Download models_{SYMBOL}.zip and extract to ml-backend/models/saved/")

## 📊 Summary

You've trained 4 models:
1. **LSTM** - Baseline sequential model
2. **Bi-LSTM** - Bidirectional (usually best)
3. **GRU** - Faster alternative to LSTM
4. **Conv1D** - Pattern recognition

**Next Steps:**
1. Download `models_BTCUSD.zip` or `models_XAUUSD.zip`
2. Extract to `ml-backend/models/saved/`
3. Start the FastAPI server: `uvicorn api.main:app --reload`
4. Test predictions in your DOM heatmap!